# Notebook 07 — Feature engineering

## Objectif
Créer de nouvelles variables susceptibles d'améliorer les performances. Chaque variable est évaluée sur les mêmes folds GroupKFold et conservée uniquement si le gain est stable.

## Fichiers utilisés
- `data/X_train.csv` — variables explicatives
- `data/y_train.csv` — variable cible

## Résultat attendu
- RMSE avant / après chaque feature
- Rapport Skore Hub (clé `06_features`)

## Notebook suivant
`08_error_analysis.ipynb` — analyse des cas où le modèle se trompe le plus.

## 1. Imports

In [7]:
import json
import os
from pathlib import Path

import pandas as pd
import skore
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold

## 2. Connexion à Skore Hub

In [8]:
config = json.loads(Path("../.skore").read_text())
os.environ["SKORE_HUB_API_KEY"] = config["api_key"]
skore.login()
hub_project = skore.Project(
    "bobathon-esilv",
    mode="hub",
    workspace=config["workspace"],
)
print(f"Connecté au Hub : workspace={hub_project.workspace}")

╭───────────────────────────────── Login to Skore Hub ─────────────────────────────────╮
│                                                                                      │
│                                  Already logged in.                                  │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Connecté au Hub : workspace=BERNIE MADOFF


## 3. Chargement et fusion des données

In [9]:
DATA_DIR = Path("../data")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")

visits = X_train.merge(
    y_train,
    on="Index",
    how="inner",
    validate="one_to_one",
)

assert len(visits) == len(X_train)
assert visits["target"].notna().all()

print(f"visits : {visits.shape} — OK")

visits : (44590, 14) — OK


## 4. Modèle de référence (HGBR, variables numériques)

In [10]:
feature_cols_base = [
    "sexM", "age_at_diagnosis", "age", "ledd",
    "time_since_intake_on", "time_since_intake_off", "on", "off",
]

y = visits["target"]
groups = visits["patient_id"]

X_base = visits[feature_cols_base]
cv_splits_base = list(GroupKFold(n_splits=5).split(X_base, y, groups=groups))

hgbr_base = HistGradientBoostingRegressor(random_state=0)
report_base = skore.evaluate(hgbr_base, X_base, y, splitter=cv_splits_base)
rmse_base = float(report_base.metrics.rmse().iloc[:, 0].values.mean())
print(f"RMSE référence (GroupKFold, moyenne) : {rmse_base:.4f}")

Output()

RMSE référence (GroupKFold, moyenne) : 7.7520


## 5. Feature 1 — Ancienneté du diagnostic

`years_since_diagnosis = age - age_at_diagnosis` : durée depuis le diagnostic. Cette variable peut capturer la progression de la maladie mieux que l'âge brut.

In [11]:
visits_f1 = visits.copy()
visits_f1["years_since_diagnosis"] = visits_f1["age"] - visits_f1["age_at_diagnosis"]

feature_cols_f1 = feature_cols_base + ["years_since_diagnosis"]
X_f1 = visits_f1[feature_cols_f1]
cv_splits_f1 = list(GroupKFold(n_splits=5).split(X_f1, y, groups=groups))

hgbr_f1 = HistGradientBoostingRegressor(random_state=0)
report_f1 = skore.evaluate(hgbr_f1, X_f1, y, splitter=cv_splits_f1)
rmse_f1 = float(report_f1.metrics.rmse().iloc[:, 0].values.mean())
print(f"RMSE + years_since_diagnosis : {rmse_f1:.4f}  (delta : {rmse_f1 - rmse_base:+.4f})")

Output()

RMSE + years_since_diagnosis : 7.6685  (delta : -0.0835)


## 6. Feature 2 — Indicateurs de valeurs manquantes

L'absence d'une mesure peut elle-même être informative : un patient sans score `off` a peut-être un profil différent.

In [12]:
visits_f2 = visits_f1.copy()

missing_cols = ["on", "off", "ledd", "time_since_intake_on", "time_since_intake_off"]
for col in missing_cols:
    visits_f2[f"{col}_missing"] = visits_f2[col].isna().astype(int)

feature_cols_f2 = feature_cols_f1 + [f"{c}_missing" for c in missing_cols]
X_f2 = visits_f2[feature_cols_f2]
cv_splits_f2 = list(GroupKFold(n_splits=5).split(X_f2, y, groups=groups))

hgbr_f2 = HistGradientBoostingRegressor(random_state=0)
report_f2 = skore.evaluate(hgbr_f2, X_f2, y, splitter=cv_splits_f2)
rmse_f2 = float(report_f2.metrics.rmse().iloc[:, 0].values.mean())
print(f"RMSE + indicateurs manquants : {rmse_f2:.4f}  (delta vs base : {rmse_f2 - rmse_base:+.4f})")

Output()

RMSE + indicateurs manquants : 7.6685  (delta vs base : -0.0835)


## 7. Feature 3 — Écart OFF − ON

`off_on_gap = off - on` : différence entre le score OFF et le score ON. Mesure l'amplitude de l'effet du traitement.

In [13]:
visits_f3 = visits_f2.copy()
visits_f3["off_on_gap"] = visits_f3["off"] - visits_f3["on"]

feature_cols_f3 = feature_cols_f2 + ["off_on_gap"]
X_f3 = visits_f3[feature_cols_f3]
cv_splits_f3 = list(GroupKFold(n_splits=5).split(X_f3, y, groups=groups))

hgbr_f3 = HistGradientBoostingRegressor(random_state=0)
report_f3 = skore.evaluate(hgbr_f3, X_f3, y, splitter=cv_splits_f3)
rmse_f3 = float(report_f3.metrics.rmse().iloc[:, 0].values.mean())
print(f"RMSE + off_on_gap            : {rmse_f3:.4f}  (delta vs base : {rmse_f3 - rmse_base:+.4f})")

Output()

RMSE + off_on_gap            : 7.6594  (delta vs base : -0.0926)


## 8. Tableau récapitulatif

In [14]:
summary = pd.DataFrame([
    {"Configuration": "Baseline (8 variables num.)", "RMSE moy.": rmse_base, "Delta": 0.0},
    {"Configuration": "+ years_since_diagnosis",     "RMSE moy.": rmse_f1,   "Delta": rmse_f1 - rmse_base},
    {"Configuration": "+ indicateurs manquants",     "RMSE moy.": rmse_f2,   "Delta": rmse_f2 - rmse_base},
    {"Configuration": "+ off_on_gap",                "RMSE moy.": rmse_f3,   "Delta": rmse_f3 - rmse_base},
])
display(summary.round(4))

,Configuration,RMSE moy.,Delta
0,Baseline (8 variables num.),7.7520,0.0000
1,+ years_since_diagnosis,7.6685,-0.0835
2,+ indicateurs manquants,7.6685,-0.0835
3,+ off_on_gap,7.6594,-0.0926


## 9. Sélection et envoi du meilleur rapport

In [15]:
# Envoyer le rapport de la configuration avec toutes les features
hub_project.put("06_features", report_f3)
print("Rapport envoyé à Skore Hub sous la clé : 06_features")
print(f"Configuration finale : {len(feature_cols_f3)} variables")
print("Variables :", feature_cols_f3)

Output()

Consult your report at https://skore.probabl.ai/BERNIE 
MADOFF/bobathon-esilv/cross-validations/61380

Rapport envoyé à Skore Hub sous la clé : 06_features
Configuration finale : 15 variables
Variables : ['sexM', 'age_at_diagnosis', 'age', 'ledd', 'time_since_intake_on', 'time_since_intake_off', 'on', 'off', 'years_since_diagnosis', 'on_missing', 'off_missing', 'ledd_missing', 'time_since_intake_on_missing', 'time_since_intake_off_missing', 'off_on_gap']


## 10. Conclusion

**Règles respectées :**
- Chaque variable calculée uniquement à partir des données disponibles (pas de `target`).
- Même transformation appliquée sur train et test (pas de fuite).
- Évaluation avec GroupKFold pour comparaison équitable.

**Prochaine étape** : `08_error_analysis.ipynb` — analyser les cas extrêmes du meilleur modèle.